## Задание 4

Обучите автоэнкодер на датасете `MNIST`, постарайтесь сделать его с размером эмбеддинга не больше 128. Сгенерируйте эмбеддинги для тысячи объектов из обучающей выборки (как в семинаре), обучите на этих эмбеддингах Случайный лес. Добейтесь Accuracy больше 90%.

Сделайте эмбеддинги для 1000 объектов из обучающей выборки, тензор с ними должен иметь размерность $(1000, N)$, где $N \leq 128$. Мы обучим на этом тензоре Случайный лес из библиотеки `scikit-learn`, а именно `RandomForestClassifier(random_state=0)`. После чего проверим качество на ваших эмбеддингах для тестовой выборки, сдайте их тоже. Тензор с эмбеддингами для тестовой выборки должен иметь размерность $(10000,N)$. Accuracy нашего обученного случайного леса должна быть больше 90% на тестовых эмбеддингах.

In [ ]:
import math

import torch
import torch.nn as nn


class Block(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, stride: int = 1, upsample: bool = False):
        super().__init__()
        self.upsample = upsample

        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=1,
            bias=False
        )
        self.norm = nn.BatchNorm2d(out_channels)
        self.act = nn.LeakyReLU(0.2)

    def forward(self, x):
        if self.upsample:
            x = nn.functional.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False, recompute_scale_factor=False)

        return self.act(self.norm(self.conv(x)))


class DenoisingBlock(Block):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, stride: int = 1, upsample: bool = False):
        super().__init__(in_channels, out_channels, kernel_size, stride, upsample)

        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        if self.upsample:
            x = nn.functional.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False, recompute_scale_factor=False)

        if self.training:
            x = x + torch.randn_like(x) * 0.05

        return self.act(self.norm(self.conv(x)))


class AutoEncoder(nn.Module):
    def __init__(self, in_channels: int, base_block: nn.Module = Block, base_size: int = 32, num_blocks: int = 4):
        super().__init__()

        self.base_size = base_size


        encoder_blocks = []

        for i in range(num_blocks):
            encoder_blocks.append(
                base_block(
                    in_channels=base_size if i else in_channels,
                    out_channels=base_size,
                    kernel_size=3,
                    stride=2
                )
            )

        encoder_blocks.append(
            base_block(
                in_channels=base_size,
                out_channels=base_size,
                kernel_size=3
            ).conv
        )

        self.encoder = nn.Sequential(*encoder_blocks) 


        decoder_blocks = []

        for i in range(num_blocks):
            decoder_blocks.append(
                base_block(
                    in_channels=base_size,
                    out_channels=base_size,
                    kernel_size=3,
                    upsample=True
                )
            )

        decoder_blocks.append(
            base_block(
                in_channels=base_size,
                out_channels=in_channels,
                kernel_size=3
            ).conv
        )
        decoder_blocks.append(nn.Sigmoid())

        self.decoder = nn.Sequential(*decoder_blocks)  

        self.flatten = nn.Flatten()

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)

        return x

    @torch.inference_mode()
    def encode(self, x):
        return self.flatten(self.encoder(x))

    @torch.inference_mode()
    def decode(self, x):
        latent_size = int(math.sqrt(x.shape[1] // self.base_size))

        return self.decoder(x.view(-1, self.base_size, latent_size, latent_size)) 


class DenoisingAutoEncoder(AutoEncoder):
    def __init__(self, in_channels: int, base_block: nn.Module = DenoisingBlock, base_size: int = 32, num_blocks: int = 4):
        super().__init__(in_channels, base_block, base_size, num_blocks)

    def forward(self, x):
        if self.training:
            x = torch.clip(x + torch.randn_like(x) * 0.1, min=0, max=1)

        x = self.encoder(x)
        x = self.decoder(x)

        return x

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as T
from IPython.display import clear_output
from torch.optim import Adam
from torch.optim import Optimizer
from torch.utils.data import DataLoader
from torch.utils.data import Subset
from torchvision.datasets import MNIST
from tqdm import tqdm

def set_seed(seed):
    torch.backends.cudnn.deterministic = True
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    

def train(
    model: nn.Module,
    data_loader: DataLoader,
    optimizer: Optimizer,
    loss_fn,
    device: torch.device,
):
    model.train()

    train_loss = 0

    for x, _ in tqdm(data_loader, desc='Train'):
        x = x.to(device)

        optimizer.zero_grad()

        output = model(x)

        loss = loss_fn(output, x)

        train_loss += loss.item()

        loss.backward()

        optimizer.step()

    train_loss /= len(data_loader)

    return train_loss


@torch.inference_mode()
def evaluate(
    model: nn.Module, data_loader: DataLoader, loss_fn, device: torch.device
):
    model.eval()

    total_loss = 0

    for x, _ in tqdm(data_loader, desc='Evaluation'):
        x = x.to(device)

        output = model(x)

        loss = loss_fn(output, x)

        total_loss += loss.item()

    total_loss /= len(data_loader)

    return total_loss


def plot_stats(
    train_loss: list[float],
    valid_loss: list[float],
    title: str
):
    plt.figure(figsize=(16, 8))

    plt.title(title + ' loss')

    plt.plot(train_loss, label='Train loss')
    plt.plot(valid_loss, label='Valid loss')
    plt.legend()
    plt.grid()

    plt.show()


def whole_train_valid_cycle(
    model, train_loader, valid_loader, optimizer, loss_fn, device, num_epochs, title
):
    train_loss_history, valid_loss_history = [], []

    for epoch in range(num_epochs):
        train_loss = train(model, train_loader, optimizer, loss_fn, device)
        valid_loss = evaluate(model, valid_loader, loss_fn, device)

        train_loss_history.append(train_loss)
        valid_loss_history.append(valid_loss)

        clear_output(wait=True)
        
        plot_stats(
            train_loss_history, valid_loss_history,
            title
        )
        

@torch.inference_mode()
def get_latent_labels(model, loader, device, count: int=None):
    model.eval()

    total = 0
    
    latent = []
    labels = []

    for x, y in loader:
        z = model.encode(x.to(device)).cpu()
        
        latent.append(z)
        labels.append(y)
        
        total += y.shape[0]
        
        if count is not None and total >= count:
            break

    latent = torch.cat(latent)
    labels = torch.cat(labels)

    if count is not None:
        latent = latent[:count]
        labels = labels[:count]

    return latent, labels


def save_embeddings(x_train, y_train, x_valid, y_valid):
    assert x_train.shape[0] == 1000
    assert x_valid.shape[0] == 10000
    
    assert y_train.shape[0] == 1000
    assert y_valid.shape[0] == 10000

    torch.save(
        {
            'x_train': x_train,
            'y_train': y_train,
            'x_valid': x_valid,
            'y_valid': y_valid
        },
        'embeddings.pt'
    )


def main(model, title):
    set_seed(0xDEADF00D)

    mnist_transforms = T.Compose(
        [
            T.Resize((64, 64)),
            T.ToTensor(),
        ]
    )

    train_dataset = MNIST('mnist', train=True, transform=mnist_transforms, download=True)
    valid_dataset = MNIST('mnist', train=False, transform=mnist_transforms, download=True)

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=8, pin_memory=True)
    valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False, num_workers=8, pin_memory=True)

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    model = model.to(device)

    optimizer = Adam(model.parameters(), lr=1e-3)

    loss_fn = nn.MSELoss()

    whole_train_valid_cycle(
        model, train_loader, valid_loader, optimizer, loss_fn, device, 7, title
    )
    
    x_train, y_train = get_latent_labels(model, train_loader, device, 1000)
    x_valid, y_valid = get_latent_labels(model, valid_loader, device)
    
    save_embeddings(x_train, y_train, x_valid, y_valid)

In [ ]:
main(DenoisingAutoEncoder(in_channels=1, base_size=8), 'Denoising AE')